# Immortal Rising on a Nothing Phone (1)

**Status:** plan, nothing built. Written 2026-10-27 after the redroid attempt failed.

**Goal:** the game (`com.badbeans.DarkIdle`) farms around the clock on an old Nothing Phone (1), and inari drives
the daily tasks over `adb`: daily quests, quick battles and shop items first; later bosses, upgrades and dungeons.

**Non-goals:** rooting the phone, unlocking the bootloader, or hiding anything from the game's integrity check.

## What the emulator attempt established

| Finding | Evidence |
|---|---|
| The game is arm64-only (Unity 6000.3, il2cpp). | The `.xapk` has only a `config.arm64_v8a` split. |
| redroid's libndk_translation crashes it. | `Could not allocate memory ... 18446744068739681312B`, then abort. |
| Houdini gets it to the title screen. | Unity initialised, `Fully drawn +10s`. |
| It then shows a **LIAPP ALERT: Rooting** and exits. | redroid is a `userdebug/test-keys` build. |
| An emulator therefore cannot run it as shipped. | LIAPP is the developers' integrity check, and we will not defeat it. |

The code written for that attempt is in git history and mostly carries over (see *What to reuse*).

## Gate: does a stock phone pass?

Everything below depends on one unknown: LIAPP may also object to **USB or wireless debugging** being on,
not only to root. Test this before building anything else. If the alert appears on the unmodified phone,
**stop**: the honest answer is that the game cannot be automated from a server, and the plan ends here.

The cells below are meant to be run from inari against the phone, with the phone on the same network.
Run them top to bottom; nothing has been executed.

In [ ]:
# Where the phone is and what to drive. Fill PHONE after pairing (see the checklist).
import subprocess

PHONE = "192.168.0.50:5555"  # placeholder: wireless-debugging address, or a USB serial from `adb devices`
PACKAGE = "com.badbeans.DarkIdle"


def adb(*args, check=True):
    result = subprocess.run(["adb", "-s", PHONE, *args], capture_output=True, text=True, check=check)
    return result.stdout.strip()

In [ ]:
# Preflight: is this a stock, unrooted phone, and is it healthy enough to leave on a charger?
subprocess.run(["adb", "connect", PHONE], check=False)
for prop in ["ro.product.model", "ro.build.tags", "ro.build.type", "ro.debuggable", "ro.build.version.release"]:
    print(prop, "=", adb("shell", "getprop", prop))
print("size:", adb("shell", "wm", "size"))
print(adb("shell", "dumpsys", "battery"))
# Expect: tags=release-keys, type=user, debuggable=0. Anything else is not a stock build.

In [ ]:
# The gate: launch the game and look at what it shows after ~20 s.
adb("shell", "monkey", "-p", PACKAGE, "-c", "android.intent.category.LAUNCHER", "1")
import time

time.sleep(20)
png = subprocess.run(["adb", "-s", PHONE, "exec-out", "screencap", "-p"], capture_output=True, check=True).stdout
open("gate.png", "wb").write(png)
print("saved gate.png: a LIAPP alert means the plan stops here; the title or login screen means it can continue")

## Phone checklist (once the gate passes)

1. Keep the stock system. No root, no unlocked bootloader, no custom ROM.
2. Install the game from Google Play and log in once by hand (Facebook or Apple, as on the iPad).
3. Turn on Play Store auto-update for the game. This replaces the `apkeep` timer: the phone updates itself.
4. Developer options: stay awake while charging; wireless debugging **or** USB debugging.
5. Settings: disable battery optimisation for the game; lock rotation to landscape if the game does not force it.
6. Give the phone a DHCP reservation on the router, so its address never changes.
7. Charging: check *Settings, Battery* for a charge limit. If there is none, put the charger on a smart plug
   and let Home Assistant (already on inari) hold the battery between about 40 and 80 percent.

**Connection choice.** Wireless debugging picks a new port whenever it is toggled or the phone reboots, so
the address in `PHONE` can go stale and needs re-pairing. A USB cable from the phone to inari is steadier and
charges at the same time, but then the battery sits at full charge unless the smart plug cuts power. Decide
after the gate passes; wireless first is easiest to try.

## What to reuse from the redroid attempt

Commits `c01f071` to `5d43416` (removed in `ba2b4a0`). Recover a file with `git show 5d43416:<path>`, with paths
under `hosts/nixos/inari/services/immortal-rising/`.

| File | Reuse |
|---|---|
| `agent.py` | The template-matching agent: `flow.json`, routines, the kicked-session grace period. Keep as is. |
| `view.py` | Browser remote. Not needed on a real phone: use the phone itself or `scrcpy`. Drop it. |
| `default.nix` | Keep the `daily` (17:30) and `periodic` (every 15 min) timers and the flock. Drop the container, image, loop mount and `apkeep` update. |
| `image.nix`, `houdini.rc` | Drop. |

**Changes for a real phone:**

- The screen is 1080 by 2400 and the game is landscape, so every template has to be captured again at that size.
- `ensure_home` should also wake and unlock the phone (`input keyevent KEYCODE_WAKEUP`, then a swipe) if there is a lock screen, or use *no lock screen* with the phone kept somewhere safe.
- The adb server runs on inari with `ANDROID_USER_HOME` under `/persist`, so the pairing key survives reboots.

## Behaviour to keep

- **Kicked session.** Logging in on another device signs this one out. The agent waits a grace period
  (`reloginGraceMinutes`, 180 by default) before signing back in, so playing on the iPad is not interrupted
  every 15 minutes.
- **One routine at a time.** `daily` and `periodic` share a lock so they never tap the same screen together.
- **Alerts.** Failures go through the n8n alert router (`POST /alert`); a grace-period wait is not a failure.
  The ntfy token for that channel is still outstanding.

## Steps

| # | Step | Done when |
|---|---|---|
| 1 | Run the gate cells on the unmodified phone | `gate.png` shows the game, not a LIAPP alert |
| 2 | Pair adb from inari and fix the address | `adb devices` lists the phone after an inari reboot |
| 3 | Add a slim NixOS module: agent, timers, `android-tools` | `just nixos-rebuild dry-build inari` passes |
| 4 | Capture templates with `agent.py shot`, write `flow.json` | `run periodic` reaches the home screen |
| 5 | Add the daily routine: quests, quick battles, shop | one full day of correct claims |
| 6 | Add the periodic routine: bosses, upgrades, dungeons | a day without manual intervention |
| 7 | Charge control through Home Assistant | battery stays inside the chosen band for a week |

## Risks

- **Account.** Automation is probably against the game's terms and could get the account banned.
- **Updates.** The game may add checks in any update that stop it working with debugging enabled.
- **Battery.** An old lithium battery on a charger around the clock can swell. Keep the phone on a hard,
  uncluttered surface, check it monthly, and replace the battery or retire the phone if it bulges.

## Clean-up still to do on inari after the switch

The module is gone from the repo, but these remain until someone with root removes them:

```sh
sudo systemctl stop docker-redroid.service     # already gone after the switch; harmless if so
sudo docker rm -f redroid
sudo docker image rm redroid-houdini:13.0.0 redroid/redroid:13.0.0-latest
sudo umount /persist/state/var/lib/immortal-rising/data 2>/dev/null
sudo rm -rf /raid5pool/immortal-rising /persist/state/var/lib/immortal-rising
```